In [51]:
import fastf1
import pandas as pd
import numpy as np
import os

cache_dir = "cache/cache_data"
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

fastf1.Cache.enable_cache(cache_dir)

session = fastf1.get_session(2023, "Monaco", "R") # Race
session.load()

laps = session.laps # DataFrame with cols
weather = session.weather_data

core           INFO 	Loading data for Monaco Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '14', '31', '44', '63', '16', '10', '55', '4', '81', '77', '21', '24', '23', '22', '11', '27', '2', '20', '18']


In [52]:
laps.columns.tolist()

['Time',
 'Driver',
 'DriverNumber',
 'LapTime',
 'LapNumber',
 'Stint',
 'PitOutTime',
 'PitInTime',
 'Sector1Time',
 'Sector2Time',
 'Sector3Time',
 'Sector1SessionTime',
 'Sector2SessionTime',
 'Sector3SessionTime',
 'SpeedI1',
 'SpeedI2',
 'SpeedFL',
 'SpeedST',
 'IsPersonalBest',
 'Compound',
 'TyreLife',
 'FreshTyre',
 'Team',
 'LapStartTime',
 'LapStartDate',
 'TrackStatus',
 'Position',
 'Deleted',
 'DeletedReason',
 'FastF1Generated',
 'IsAccurate']

### **DataFrame Cleaning**

In [53]:
laps['TyreLife'].head()
df = laps.copy()

df = df.dropna(subset=['LapTime'])

df['LapTime'] = df['LapTime'].dt.total_seconds()

### **Filter out non-representative laps**
- make sure to explore the race_control_message for better model fit

In [ ]:
# remove lap 1 of each race
df = df[df['LapNumber'] != 1]

# remove in-laps & out laps (lap where driver pits and after pitting)
df = df[df['PitInTime'].isna()] # remove in-laps
df = df[df['PitOutTime'].isna()] # remove out-laps

# remove laps under safety car, yellow, red, flag
df = df[df['TrackStatus'] == '1'] # keeping only clean laps

### **Fuel Correction per lap**


In [55]:
df['LapNumber'].max()

np.float64(78.0)

### **Dirty air Flagging**
Concept:
- When an F1 car is within roughly 1.5 seconds of the car ahead, it loses aerodynamic downforce from the turbulent air (dirty air or wake). This makes the car slower, particularly in speed corners. The lap time is artifically inflated, not because of tires, but because of aerodynamics. We want to flag these laps so we can either exclude them from the degradation model or treat them separately.

Pseudocode:
```
    For each lap_number in the race:
        1. Get all drivers' data for that lap
        2. Sort by Position (track position, P1 first)
        3. For each driver (except P1):
            - Find the driver one position ahead
            - gap = this_driver.LapStartDate - ahead_driver.LapStartDate
            - Convert gap to seconds
            - If gap < 1.5s → flag as dirty air
        4. P1 is never in dirty air (no one ahead)
```


In [56]:
# sort by lap number and track position
df = df.sort_values(['LapNumber', 'Position'])

# get lapstartdate of the car directly ahead (one position higher)
ahead_start = df.groupby('LapNumber')['LapStartDate'].shift(1)

# compute gap in seconds
gap_to_ahead = (df['LapStartDate'] - ahead_start).dt.total_seconds()

# flag dirty air: gap < 1.5s (and not null, which handles p1)
df['DirtyAir'] = gap_to_ahead < 1.5

In [57]:
df['DirtyAir'].tail()

1204     True
1360    False
387     False
155      True
1282     True
Name: DirtyAir, dtype: bool

### **Fuel Compenstation**
- we want to compensate the fuel loss throughout the race as a normalization technique. 
- minimize the influence of fuel weight on laptimes, so that tire degradation becomes the dominant visible signle

In [58]:
starting_fuel = 110 # est in kg
total_race_laps = session.total_laps
fuel_per_lap = starting_fuel / total_race_laps
fuel_time_per_kg = 0.035 # s/kg


# laps_remaining = total_race_laps - df['LapNumber'] 
fuel_correction = df['LapNumber'] * fuel_per_lap * fuel_time_per_kg # how much fuel has been burned, the advantage to compensate for

df['FuelCorrectedLapTime'] = df['LapTime'] + fuel_correction

In [59]:
df['FuelCorrectedLapTime'].head()

1       79.465718
233     79.744718
973     80.399718
1206    80.691718
1128    80.625718
Name: FuelCorrectedLapTime, dtype: float64

### **Output clean DataFrame**

In [60]:
cols_to_keep = [
    'Driver',
    'LapNumber',
    'Stint',
    'Compound',
    'TyreLife',
    'Team',
    'Position',
    'DirtyAir',
    'LapTime',
    'FuelCorrectedLapTime'
]

df = df[cols_to_keep]

In [61]:
df.head()

,Driver,LapNumber,Stint,Compound,TyreLife,Team,Position,DirtyAir,LapTime,FuelCorrectedLapTime
1,VER,2.0,1.0,MEDIUM,2.0,Red Bull Racing,1.0,False,79.367,79.465718
233,ALO,2.0,1.0,HARD,3.0,Aston Martin,2.0,True,79.646,79.744718
973,OCO,2.0,1.0,MEDIUM,2.0,Alpine,3.0,True,80.301,80.399718
1206,SAI,2.0,1.0,HARD,2.0,Ferrari,4.0,True,80.593,80.691718
1128,HAM,2.0,1.0,MEDIUM,2.0,Mercedes,5.0,True,80.527,80.625718
